In [54]:
from supabase import create_client
from dotenv import load_dotenv
import os, pandas as pd

load_dotenv()
supabase = create_client(os.getenv("SUPABASE_URL"), os.getenv("SUPABASE_KEY"))
df_orders = pd.DataFrame(supabase.table('sales').select('*').execute().data)
df_customers = pd.DataFrame(supabase.table('customers').select('*').execute().data)
print(f"Tellimusi: {len(df_orders)}, Kliente: {len(df_customers)}")
print(df_orders.dtypes)

2026-06-21 21:04:52,634 - INFO - HTTP Request: GET https://eocuprgglufoalmnphvi.supabase.co/rest/v1/sales?select=%2A "HTTP/2 200 OK"
2026-06-21 21:04:52,734 - INFO - HTTP Request: GET https://eocuprgglufoalmnphvi.supabase.co/rest/v1/customers?select=%2A "HTTP/2 200 OK"


Tellimusi: 1000, Kliente: 1000
id                  int64
sale_id             int64
invoice_id            str
sale_date             str
customer_id       float64
product_id          int64
quantity            int64
unit_price        float64
total_price       float64
channel               str
store_location        str
payment_method        str
dtype: object


In [55]:
# Variant A (Supabase): .select('*').eq('store_location', 'Tallinn').order('total_price', desc=True)
# Variant B (pandas):
df_tallinn = df_orders.merge(df_customers, on='customer_id')
df_tallinn = df_tallinn[df_tallinn['store_location'] == 'Tallinn'].sort_values('total_price', ascending=False)
print(f"Tallinna tellimusi: {len(df_tallinn)}, Käive: {df_tallinn['total_price'].sum():.2f} EUR")

Tallinna tellimusi: 102, Käive: 30022.00 EUR


In [56]:
df_tartu = df_orders.merge(df_customers, on='customer_id')
df_tartu = df_tartu[df_tartu['store_location'] == 'Tartu'].sort_values('total_price', ascending=False)
print(f"Tartu tellimusi: {len(df_tartu)}, Käive: {df_tartu['total_price'].sum():.2f} EUR, Keskmine tellimus: {df_tartu['total_price'].mean():.2f} EUR")

Tartu tellimusi: 52, Käive: 15564.57 EUR, Keskmine tellimus: 299.32 EUR


In [57]:
df_suured = df_orders[df_orders['total_price'] > 100]
print(f"Suured tellimused: {df_suured.shape}")
print(df_suured.describe())

Suured tellimused: (796, 12)
               id     sale_id  customer_id   product_id    quantity  \
count   796.00000   796.00000   741.000000   796.000000  796.000000   
mean    497.66206   497.66206  3523.105263  1174.296482    1.918342   
std     287.71153   287.71153   839.148959    99.648982    1.043514   
min       1.00000     1.00000  2014.000000  1001.000000    1.000000   
25%     248.75000   248.75000  2806.000000  1089.000000    1.000000   
50%     494.50000   494.50000  3556.000000  1174.500000    2.000000   
75%     746.25000   746.25000  4214.000000  1258.250000    2.000000   
max    1000.00000  1000.00000  4989.000000  1350.000000    5.000000   

       unit_price  total_price  
count  796.000000   796.000000  
mean   189.483505   349.837475  
std     84.379017   259.069133  
min     24.670000   100.400000  
25%    120.170000   176.160000  
50%    183.310000   267.945000  
75%    253.260000   436.840000  
max    434.080000  1792.050000  


In [58]:
df_orders['sale_date'] = pd.to_datetime(df_orders['sale_date'])
cutoff = df_orders['sale_date'].max() - pd.Timedelta(days=30)
df_recent = df_orders[df_orders['sale_date'] >= cutoff]
print(f"Viimase 30 päeva tellimused: {df_recent.shape}")
print(df_recent.head())

Viimase 30 päeva tellimused: (1, 12)
      id  sale_id        invoice_id  sale_date  customer_id  product_id  \
933  934      934  INV-202304-00051 2026-03-17       4113.0        1082   

     quantity  unit_price  total_price channel store_location payment_method  
933         3      100.93       302.79    pood        Tallinn          kaart  


In [59]:
klient_id = df_orders['customer_id'].iloc[0]  # võtab esimese kliendi
df_klient = df_orders[df_orders['customer_id'] == klient_id]
print(f"Kliendi {klient_id} ostud: {df_klient.shape}")
print(df_klient[['sale_date', 'total_price', 'store_location']])

Kliendi 2588.0 ostud: (1, 12)
   sale_date  total_price store_location
0 2023-01-10       469.58        Tallinn


In [60]:
def city_report(df, city):
    """Genereeri asukohapõhine müügiraport."""
    city_data = df[df['store_location'] == city]
    return {'store_location': city, 'orders': len(city_data),
            'revenue': city_data['total_price'].sum()}

for city in ['Tallinn', 'Tartu', 'Parnu']:
    r = city_report(df_orders, city)
    print(f"{r['store_location']}: {r['orders']} tellimust, {r['revenue']:.2f} EUR")


Tallinn: 398 tellimust, 112060.36 EUR
Tartu: 175 tellimust, 51383.15 EUR
Parnu: 0 tellimust, 0.00 EUR


In [61]:
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def safe_fetch(supabase_client, table_name):
    """Too andmed turvalise vigade käsitlemisega."""
    try:
        response = supabase_client.table(table_name).select('*').execute()
        df = pd.DataFrame(response.data)
        if len(df) == 0:
            logger.warning(f"Tabel '{table_name}' on tühi!")
        logger.info(f"Laaditud {len(df)} rida tabelist '{table_name}'")
        return df
    except Exception as e:
        logger.error(f"Viga tabeli '{table_name}' lugemisel: {e}")
        return pd.DataFrame()  # Tagasta tühi DataFrame vea korral

In [62]:
all_data = []
offset = 0
while True:
    response = supabase.table('sales').select('*').range(offset, offset + 999).execute()
    if not response.data:
        break
    all_data.extend(response.data)
    offset += 1000

df_orders = pd.DataFrame(all_data)
print(f"Laaditud {len(df_orders)} tellimust")

from datetime import datetime

def weekly_sales_report(df, report_date=None):
    """Genereeri iganädalane müügiraport.

    Args:
        df: DataFrame müügitellimustega
        report_date: Raporti kuupäev (vaikimisi täna)
    Returns:
        dict: Raporti kokkuvõte
    """
    if report_date is None:
        report_date = datetime.now().strftime('%Y-%m-%d')
    return {
        'report_date': report_date,
        'total_orders': len(df),
        'total_revenue': round(df['total_price'].sum(), 2),
        'avg_order': round(df['total_price'].mean(), 2),
    }

# Käivita
result = weekly_sales_report(df_orders)
for key, value in result.items():
    print(f"  {key}: {value}")

2026-06-21 21:04:53,023 - INFO - HTTP Request: GET https://eocuprgglufoalmnphvi.supabase.co/rest/v1/sales?select=%2A&offset=0&limit=1000 "HTTP/2 200 OK"
2026-06-21 21:04:53,123 - INFO - HTTP Request: GET https://eocuprgglufoalmnphvi.supabase.co/rest/v1/sales?select=%2A&offset=1000&limit=1000 "HTTP/2 200 OK"
2026-06-21 21:04:53,259 - INFO - HTTP Request: GET https://eocuprgglufoalmnphvi.supabase.co/rest/v1/sales?select=%2A&offset=2000&limit=1000 "HTTP/2 200 OK"
2026-06-21 21:04:53,791 - INFO - HTTP Request: GET https://eocuprgglufoalmnphvi.supabase.co/rest/v1/sales?select=%2A&offset=3000&limit=1000 "HTTP/2 200 OK"
2026-06-21 21:04:53,904 - INFO - HTTP Request: GET https://eocuprgglufoalmnphvi.supabase.co/rest/v1/sales?select=%2A&offset=4000&limit=1000 "HTTP/2 200 OK"
2026-06-21 21:04:54,047 - INFO - HTTP Request: GET https://eocuprgglufoalmnphvi.supabase.co/rest/v1/sales?select=%2A&offset=5000&limit=1000 "HTTP/2 200 OK"
2026-06-21 21:04:54,160 - INFO - HTTP Request: GET https://eocuprgg

Laaditud 10118 tellimust
  report_date: 2026-06-21
  total_orders: 10118
  total_revenue: 2909177.98
  avg_order: 287.53


In [63]:
import pandas as pd

def calculate_rfm(df, reference_date=None):
    """Arvuta RFM skoorid ja segmendid."""
    if reference_date is None:
        reference_date = pd.to_datetime('today')
    else:
        reference_date = pd.to_datetime(reference_date)

    df = df.copy()
    df['sale_date'] = pd.to_datetime(df['sale_date'])

    # Recency
    recency = df.groupby('customer_id')['sale_date'].max().reset_index()
    recency.columns = ['customer_id', 'last_purchase']
    recency['recency_days'] = (reference_date - recency['last_purchase']).dt.days

    # Frequency
    frequency = df.groupby('customer_id').size().reset_index(name='frequency')

    # Monetary
    monetary = df.groupby('customer_id')['total_price'].sum().reset_index()
    monetary.columns = ['customer_id', 'monetary']

    # Liida kokku
    rfm = recency[['customer_id', 'recency_days']].merge(
        frequency, on='customer_id'
    ).merge(
        monetary, on='customer_id'
    )

    # Skoorid
    rfm['R_score'] = pd.qcut(rfm['recency_days'], q=3, labels=[3, 2, 1]).astype(int)
    rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), q=3, labels=[1, 2, 3]).astype(int)
    rfm['M_score'] = pd.qcut(rfm['monetary'], q=3, labels=[1, 2, 3]).astype(int)
    rfm['RFM_score'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']

    # Segmendid
    def assign_segment(score):
        if score >= 8:
            return 'VIP Champions'
        elif score >= 6:
            return 'Loyal'
        elif score >= 4:
            return 'Potential'
        else:
            return 'At Risk'

    rfm['segment'] = rfm['RFM_score'].apply(assign_segment)
    return rfm

# Testi
rfm_result = calculate_rfm(df_orders, reference_date='2024-08-01')
print(rfm_result.sort_values('RFM_score', ascending=False))
print(f"\nSegmentide jaotus:")
print(rfm_result['segment'].value_counts())

      customer_id  recency_days  frequency  monetary  R_score  F_score  \
2205       4599.0          -194          6   1261.55        3        3   
275        2318.0          -183         11   3148.80        3        3   
288        2335.0          -129          5   1602.87        3        3   
753        2889.0          -654         74  23385.82        3        3   
1837       4160.0          -170          7   3559.39        3        3   
...           ...           ...        ...       ...      ...      ...   
870        3032.0           511          1    205.89        1        1   
918        3089.0            63          1    160.21        1        1   
921        3093.0           446          1    153.62        1        1   
927        3100.0           291          2    364.24        1        1   
803        2950.0           171          1    410.00        1        1   

      M_score  RFM_score        segment  
2205        3          9  VIP Champions  
275         3          9  V

In [64]:
def top_products_report(df, n=10):
    """Tagasta TOP N toodet käibe järgi."""
    result = (df.groupby('product_id')['total_price']
              .sum()
              .sort_values(ascending=False)
              .head(n)
              .reset_index())
    result.columns = ['product_id', 'revenue']
    return result

def churn_risk_alert(df, days_threshold=60):
    """Tagasta kliendid, kes pole N päeva ostnud."""
    df = df.copy()
    df['sale_date'] = pd.to_datetime(df['sale_date'])
    last_purchase = df.groupby('customer_id')['sale_date'].max().reset_index()
    last_purchase['days_since'] = (df['sale_date'].max() - last_purchase['sale_date']).dt.days
    at_risk = last_purchase[last_purchase['days_since'] >= days_threshold]
    return at_risk.sort_values('days_since', ascending=False)

def city_comparison(df, cities=['Tallinn', 'Tartu']):
    """Võrdle linnade müüginäitajaid."""
    result = []
    for city in cities:
        city_df = df[df['store_location'] == city]
        result.append({
            'city': city,
            'orders': len(city_df),
            'revenue': round(city_df['total_price'].sum(), 2),
            'avg_order': round(city_df['total_price'].mean(), 2)
        })
    return pd.DataFrame(result)

# Testi
print(top_products_report(df_orders, n=5))
print(churn_risk_alert(df_orders, days_threshold=30))
print(city_comparison(df_orders, cities=['Tallinn', 'Tartu', 'Parnu']))

   product_id   revenue
0        1031  27347.04
1        1108  23376.15
2        1027  22188.80
3        1006  22039.98
4        1049  21309.96
      customer_id  sale_date  days_since
573        2677.0 2023-01-07        1226
2076       4444.0 2023-01-14        1219
2336       4745.0 2023-01-20        1213
320        2372.0 2023-01-24        1209
105        2122.0 2023-02-07        1195
...           ...        ...         ...
149        2178.0 2026-03-11          67
344        2400.0 2026-03-12          66
1797       4113.0 2026-03-17          61
1619       3897.0 2026-03-27          51
1853       4178.0 2026-04-02          45

[2545 rows x 3 columns]
      city  orders     revenue  avg_order
0  Tallinn    3801  1092083.15     287.31
1    Tartu    1797   521603.11     290.26
2    Parnu       0        0.00        NaN


In [65]:
import pandas as pd
import plotly.express as px
from datetime import datetime

# === EXTRACT ===
def extract_orders():
    """Simuleeri andmete toomist API-st."""
    print("[EXTRACT] Laadin...")
    data = {'customer_id': [1001,1002,1003,1001,1002,1004,1003,1001,1005,1004,
                            1002,1003,1005,1001,1006,1004,1002,1007,1003,1005],
            'sale_date': pd.date_range('2024-01-15', periods=20, freq='10D'),
            'total_price': [89.99,45.50,120.00,67.30,55.00,210.00,33.50,145.00,
                            78.00,92.00,160.00,44.00,88.50,230.00,37.00,175.00,
                            110.00,65.00,95.00,125.00],
            'store_location': ['Tallinn','Tartu','Tallinn','Tallinn','Tartu','Parnu','Tallinn',
                     'Tallinn','Tartu','Parnu','Tartu','Tallinn','Tartu','Tallinn',
                     'Parnu','Parnu','Tartu','Tallinn','Tallinn','Tartu']}
    df = pd.DataFrame(data)
    print(f"[EXTRACT] {len(df)} tellimust laaditud")
    return df

# === TRANSFORM ===
def transform_monthly(df):
    """Arvuta kuuraport."""
    print("[TRANSFORM] Arvutan...")
    monthly = df.groupby(df['sale_date'].dt.to_period('M')).agg(
        tellimusi=('sale_date', 'count'), kaive=('total_price', 'sum')
    ).reset_index()
    monthly['sale_date'] = monthly['sale_date'].astype(str)
    monthly['kaive'] = monthly['kaive'].round(2)
    print(f"[TRANSFORM] {len(monthly)} kuud")
    return monthly

# === LOAD ===
def load_report(monthly):
    """Salvesta CSV ja graafik."""
    ts = datetime.now().strftime('%Y%m%d_%H%M')
    monthly.to_csv(f'monthly_report_{ts}.csv', index=False)
    px.bar(monthly, x='sale_date', y='kaive',
           title=f'UrbanStyle kuukäive ({ts})',
           labels={'sale_date': 'Kuu', 'kaive': 'Käive (EUR)'}
    ).write_html(f'monthly_chart_{ts}.html')
    print(f"[LOAD] CSV + HTML salvestatud")

# === RUN ===
print("PIPELINE START")
df = extract_orders()
monthly = transform_monthly(df)
load_report(monthly)
print("PIPELINE COMPLETE")
print(monthly.to_string(index=False))


PIPELINE START
[EXTRACT] Laadin...
[EXTRACT] 20 tellimust laaditud
[TRANSFORM] Arvutan...
[TRANSFORM] 7 kuud
[LOAD] CSV + HTML salvestatud
PIPELINE COMPLETE
sale_date  tellimusi  kaive
  2024-01          2 135.49
  2024-02          3 242.30
  2024-03          3 388.50
  2024-04          3 330.00
  2024-05          3 362.50
  2024-06          3 322.00
  2024-07          3 285.00


In [66]:
import pandas as pd
import plotly.express as px
from datetime import datetime


def extract_orders():
    data = {
        'customer_id': [1001,1002,1003,1001,1002,1004,1003,1001,1005,1004,
                        1002,1003,1005,1001,1006,1004,1002,1007,1003,1005],
        'sale_date': pd.date_range('2024-01-15', periods=20, freq='10D'),
        'total_price': [89.99,45.50,120.00,67.30,55.00,210.00,33.50,145.00,
                        78.00,92.00,160.00,44.00,88.50,230.00,37.00,175.00,
                        110.00,65.00,95.00,125.00],
        'store_location': ['Tallinn','Tartu','Tallinn','Tallinn','Tartu','Parnu',
                           'Tallinn','Tallinn','Tartu','Parnu','Tartu','Tallinn',
                           'Tartu','Tallinn','Parnu','Parnu','Tartu','Tallinn',
                           'Tallinn','Tartu']
    }
    return pd.DataFrame(data)

def transform_rfm(df, reference_date='2024-08-01'):
    ref = pd.to_datetime(reference_date)
    rfm = df.groupby('customer_id').agg(
        last_purchase=('sale_date', 'max'),
        frequency=('sale_date', 'count'),
        monetary=('total_price', 'sum')
    ).reset_index()
    rfm['recency_days'] = (ref - rfm['last_purchase']).dt.days
    rfm['segment'] = rfm.apply(
        lambda row: 'VIP' if row['monetary'] > 200 and row['frequency'] >= 3
                    else 'Loyal' if row['frequency'] >= 3
                    else 'At Risk' if row['recency_days'] > 120
                    else 'Regular', axis=1)
    return rfm

In [67]:
df = extract_orders()
rfm = transform_rfm(df)
print(rfm.to_string(index=False))

 customer_id last_purchase  frequency  monetary  recency_days segment
        1001    2024-05-24          4    532.29            69     VIP
        1002    2024-06-23          4    370.50            39     VIP
        1003    2024-07-13          4    292.50            19     VIP
        1004    2024-06-13          3    477.00            49     VIP
        1005    2024-07-23          3    291.50             9     VIP
        1006    2024-06-03          1     37.00            59 Regular
        1007    2024-07-03          1     65.00            29 Regular


In [68]:
import pandas as pd
import plotly.express as px
from datetime import datetime

# ============================================================
# MARKO IGANADALANE RFM PIPELINE
# ============================================================

# --- EXTRACT ---
def extract():
    """Too andmed (asenda Supabase API-ga, kui ühendus olemas)."""
    print(f"[EXTRACT] Alustan...")
    orders = pd.DataFrame({
        'customer_id': [1001,1002,1003,1001,1002,1004,1003,1001,1005,1004,
                        1002,1003,1005,1001,1006,1004,1002,1007,1003,1005],
        'sale_date': pd.date_range('2024-01-15', periods=20, freq='10D'),
        'total_price': [89.99,45.50,120.00,67.30,55.00,210.00,33.50,145.00,
                        78.00,92.00,160.00,44.00,88.50,230.00,37.00,175.00,
                        110.00,65.00,95.00,125.00],
        'store_location': ['Tallinn','Tartu','Tallinn','Tallinn','Tartu','Parnu',
                 'Tallinn','Tallinn','Tartu','Parnu','Tartu','Tallinn',
                 'Tartu','Tallinn','Parnu','Parnu','Tartu','Tallinn',
                 'Tallinn','Tartu']
    })
    customers = pd.DataFrame({
        'customer_id': [1001,1002,1003,1004,1005,1006,1007],
        'first_name': ['Juri','Kati','Maris','Peeter','Liina','Andres','Tiina'],
        'last_name': ['Tamm','Kask','Sepp','Rebane','Ots','Puu','Kuusk']
    })
    print(f"[EXTRACT] {len(orders)} tellimust, {len(customers)} klienti")
    return orders, customers

# --- TRANSFORM ---
def transform(orders, customers, reference_date='2024-08-01'):
    """Puhasta, arvuta RFM, loo kuuraport."""
    print(f"[TRANSFORM] Alustan...")
    ref = pd.to_datetime(reference_date)
    df = pd.merge(orders, customers, on='customer_id', how='left')

    # Kuuraport
    monthly = df.groupby(df['sale_date'].dt.to_period('M')).agg(
        tellimusi=('sale_date', 'count'), kaive=('total_price', 'sum')
    ).reset_index()
    monthly['sale_date'] = monthly['sale_date'].astype(str)

    # RFM
    rfm = df.groupby('customer_id').agg(
        last_purchase=('sale_date', 'max'), frequency=('sale_date', 'count'),
        monetary=('total_price', 'sum'), nimi=('first_name', 'first')
    ).reset_index()
    rfm['recency_days'] = (ref - rfm['last_purchase']).dt.days
    rfm['segment'] = rfm.apply(
        lambda r: 'VIP' if r['monetary'] > 300 and r['frequency'] >= 3
                  else 'Loyal' if r['frequency'] >= 3
                  else 'At Risk' if r['recency_days'] > 120
                  else 'Regular', axis=1)

    print(f"[TRANSFORM] {len(rfm)} klienti segmenteeritud")
    return {'monthly': monthly, 'rfm': rfm}

# --- VALIDATE ---
def validate(results):
    """Kontrolli andmete kvaliteeti."""
    ok = len(results['rfm']) > 0 and results['monthly']['kaive'].sum() > 0
    print(f"[VALIDATE] {'OK' if ok else 'PROBLEEM!'}")
    return ok

# --- LOAD ---
def load(results):
    """Salvesta CSV ja graafikud."""
    ts = datetime.now().strftime('%Y%m%d')
    results['rfm'].to_csv(f'rfm_report_{ts}.csv', index=False)
    px.scatter(results['rfm'], x='recency_days', y='monetary', color='segment',
               size='frequency', hover_data=['nimi'],
               title=f'UrbanStyle RFM ({ts})',
               labels={'recency_days': 'Paevi viimasest ostust', 'monetary': 'Kogukulutus (EUR)'}
    ).write_html(f'rfm_chart_{ts}.html')
    px.bar(results['monthly'], x='sale_date', y='kaive',
           title=f'Kuukäive ({ts})', labels={'sale_date': 'Kuu', 'kaive': 'EUR'}
    ).write_html(f'monthly_chart_{ts}.html')
    print(f"[LOAD] 1 CSV + 2 HTML salvestatud")

# --- RUN ---
print("=" * 50)
print("  MARKO IGANADALANE RFM PIPELINE")
print("=" * 50)
start = datetime.now()
orders, customers = extract()
results = transform(orders, customers)
if validate(results):
    load(results)
print(f"  VALMIS ({(datetime.now() - start).total_seconds():.1f}s)")
print("=" * 50)

# Kokkuvõte
print("\n--- SEGMENDID ---")
print(results['rfm'][['nimi', 'frequency', 'monetary', 'segment']]
      .sort_values('monetary', ascending=False).to_string(index=False))

  MARKO IGANADALANE RFM PIPELINE
[EXTRACT] Alustan...
[EXTRACT] 20 tellimust, 7 klienti
[TRANSFORM] Alustan...
[TRANSFORM] 7 klienti segmenteeritud
[VALIDATE] OK
[LOAD] 1 CSV + 2 HTML salvestatud
  VALMIS (0.2s)

--- SEGMENDID ---
  nimi  frequency  monetary segment
  Juri          4    532.29     VIP
Peeter          3    477.00     VIP
  Kati          4    370.50     VIP
 Maris          4    292.50   Loyal
 Liina          3    291.50   Loyal
 Tiina          1     65.00 Regular
Andres          1     37.00 Regular
